# MCMO Adapter - Teste Completo

Notebook para testar o MCMOAdapter com dados reais.

**Estratégia:**
1. Conectar Google Drive e acessar pasta `mcmo/`
2. Instalar dependências (geatpy, scikit-multiflow)
3. Testar adapter com dados sintéticos
4. Testar com Electricity dataset
5. Comparar com baseline simples
6. Gerar relatório de resultados

**Autor:** Claude Code  
**Data:** 2025-11-24

## 1. Setup - Conectar Google Drive

In [ ]:
from google.colab import drive
import os
import sys

# Montar Google Drive
drive.mount('/content/drive')

# Definir path para a pasta DSL-AG-hybrid
# AJUSTE O PATH ABAIXO para o local correto no seu Google Drive
DSL_PATH = '/content/drive/MyDrive/DSL-AG-hybrid'  # Ajustar conforme necessário

# Verificar se a pasta existe
if os.path.exists(DSL_PATH):
    print(f"✓ Pasta encontrada: {DSL_PATH}")
    sys.path.insert(0, DSL_PATH)
    print(f"✓ Path adicionado ao sys.path")
else:
    print(f"✗ Pasta não encontrada: {DSL_PATH}")
    print("\nPor favor, faça upload da pasta DSL-AG-hybrid para o Google Drive")
    print("ou ajuste o caminho DSL_PATH na célula acima.")

# Verificar se a pasta mcmo existe
MCMO_PATH = os.path.join(DSL_PATH, 'mcmo')
if os.path.exists(MCMO_PATH):
    print(f"✓ Pasta mcmo encontrada: {MCMO_PATH}")
    files = os.listdir(MCMO_PATH)
    print(f"  Arquivos: {files}")
else:
    print(f"✗ Pasta mcmo não encontrada: {MCMO_PATH}")

## 2. Instalar Dependências

In [ ]:
%%capture install_output

# Instalar dependências do MCMO
!pip install geatpy==2.7.0
!pip install scikit-multiflow==0.5.3

print("Dependências instaladas!")

In [ ]:
# Verificar instalação
import warnings
warnings.filterwarnings('ignore')

deps_ok = True

try:
    import geatpy as ea
    print("✓ geatpy instalado")
except ImportError as e:
    print(f"✗ geatpy: {e}")
    deps_ok = False

try:
    import skmultiflow
    print("✓ scikit-multiflow instalado")
except ImportError as e:
    print(f"✗ scikit-multiflow: {e}")
    deps_ok = False

try:
    import numpy as np
    import pandas as pd
    from sklearn.metrics import accuracy_score, classification_report
    print("✓ numpy, pandas, sklearn disponíveis")
except ImportError as e:
    print(f"✗ Outras dependências: {e}")
    deps_ok = False

if deps_ok:
    print("\n✓ Todas as dependências OK!")
else:
    print("\n✗ Algumas dependências faltando. Execute a célula de instalação novamente.")

## 3. Importar MCMOAdapter

In [ ]:
# Importar adapter
try:
    from mcmo.baseline_mcmo import MCMOAdapter, MCMOEvaluator, test_mcmo_adapter
    print("✓ MCMOAdapter importado com sucesso!")
    
    # Verificar se MCMO está disponível
    from mcmo import baseline_mcmo
    if baseline_mcmo.MCMO_AVAILABLE:
        print("✓ MCMO disponível e pronto para uso")
    else:
        print(f"✗ MCMO não disponível: {baseline_mcmo.IMPORT_ERROR}")
        
except ImportError as e:
    print(f"✗ Erro ao importar: {e}")
    print("\nVerifique:")
    print("1. A pasta mcmo está no Google Drive")
    print("2. O path DSL_PATH está correto")
    print("3. As dependências foram instaladas")

## 4. Teste com Dados Sintéticos

In [ ]:
import numpy as np
from sklearn.datasets import make_classification

print("=" * 70)
print("Teste 1: Dados Sintéticos")
print("=" * 70)

# Gerar dataset sintético com drift
np.random.seed(42)

n_samples_per_chunk = 500
n_chunks = 10
n_features = 10

print(f"\nGerando {n_chunks} chunks de {n_samples_per_chunk} amostras cada...")

X_chunks = []
y_chunks = []

for i in range(n_chunks):
    # Adicionar drift gradual mudando os pesos das classes
    weights = [0.5 + 0.05*i, 0.5 - 0.05*i]
    
    X, y = make_classification(
        n_samples=n_samples_per_chunk,
        n_features=n_features,
        n_informative=8,
        n_redundant=2,
        n_classes=2,
        weights=weights,
        flip_y=0.01,
        random_state=42+i
    )
    
    X_chunks.append(X)
    y_chunks.append(y)
    
    print(f"  Chunk {i+1}: {len(X)} samples, class distribution: {np.bincount(y)}")

print("\n✓ Dataset sintético gerado com drift gradual")

In [ ]:
# Testar MCMOAdapter
print("\n" + "=" * 70)
print("Testando MCMOAdapter com n_sources=3")
print("=" * 70 + "\n")

results = test_mcmo_adapter(
    X_chunks=X_chunks,
    y_chunks=y_chunks,
    n_sources=3,
    verbose=True
)

print("\n" + "=" * 70)
print("Resultados Finais - Dados Sintéticos")
print("=" * 70)
print(f"Acurácia Média: {results['mean_accuracy']:.4f}")
print(f"Total de Amostras: {results['global_metrics']['total_samples']}")
print(f"Chunks Processados: {results['global_metrics']['total_chunks_processed']}")

# Plot accuracies por chunk
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))
plt.plot(range(1, len(results['accuracies'])+1), results['accuracies'], 
         marker='o', linewidth=2, markersize=8)
plt.axhline(y=results['mean_accuracy'], color='r', linestyle='--', 
            label=f'Mean: {results["mean_accuracy"]:.4f}')
plt.xlabel('Chunk', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('MCMO Adapter - Accuracy per Chunk (Synthetic Data)', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Teste com Electricity Dataset

In [ ]:
print("=" * 70)
print("Teste 2: Electricity Dataset")
print("=" * 70)

# Tentar carregar Electricity dataset
electricity_paths = [
    os.path.join(DSL_PATH, 'datasets', 'Electricity.csv'),
    os.path.join(DSL_PATH, 'data', 'Electricity.csv'),
    '/content/drive/MyDrive/datasets/Electricity.csv',
]

electricity_data = None
for path in electricity_paths:
    if os.path.exists(path):
        print(f"\n✓ Carregando de: {path}")
        electricity_data = pd.read_csv(path)
        break

if electricity_data is None:
    print("\n✗ Electricity dataset não encontrado.")
    print("\nTentou em:")
    for path in electricity_paths:
        print(f"  - {path}")
    print("\nVocê pode:")
    print("1. Fazer upload do Electricity.csv para uma das pastas acima")
    print("2. Usar dataset alternativo (SEA, Agrawal, etc.)")
    print("3. Pular este teste e usar apenas dados sintéticos")
else:
    print(f"\n✓ Dataset carregado: {electricity_data.shape}")
    print(f"  Colunas: {list(electricity_data.columns)}")
    print(f"\nPrimeiras linhas:")
    print(electricity_data.head())

In [ ]:
if electricity_data is not None:
    # Preparar dados
    print("\nPreparando dados...")
    
    # Assumir que última coluna é o target
    # Ajustar conforme necessário
    target_col = 'class' if 'class' in electricity_data.columns else electricity_data.columns[-1]
    
    X_elec = electricity_data.drop(target_col, axis=1).values
    y_elec = electricity_data[target_col].values
    
    # Se labels são strings, converter para numérico
    if y_elec.dtype == object:
        from sklearn.preprocessing import LabelEncoder
        le = LabelEncoder()
        y_elec = le.fit_transform(y_elec)
        print(f"  Labels convertidos: {le.classes_}")
    
    print(f"\n✓ X shape: {X_elec.shape}")
    print(f"✓ y shape: {y_elec.shape}")
    print(f"✓ Classes: {np.unique(y_elec)}")
    print(f"✓ Class distribution: {np.bincount(y_elec)}")
    
    # Dividir em chunks
    chunk_size = 1000
    n_chunks_elec = len(X_elec) // chunk_size
    
    print(f"\nDividindo em chunks de {chunk_size} amostras...")
    print(f"Total de chunks: {n_chunks_elec}")
    
    X_chunks_elec = []
    y_chunks_elec = []
    
    for i in range(n_chunks_elec):
        start = i * chunk_size
        end = start + chunk_size
        X_chunks_elec.append(X_elec[start:end])
        y_chunks_elec.append(y_elec[start:end])
    
    print(f"✓ {len(X_chunks_elec)} chunks criados")

In [ ]:
if electricity_data is not None:
    print("\n" + "=" * 70)
    print("Testando MCMOAdapter em Electricity")
    print("=" * 70 + "\n")
    
    results_elec = test_mcmo_adapter(
        X_chunks=X_chunks_elec,
        y_chunks=y_chunks_elec,
        n_sources=3,
        verbose=True
    )
    
    print("\n" + "=" * 70)
    print("Resultados Finais - Electricity Dataset")
    print("=" * 70)
    print(f"Acurácia Média: {results_elec['mean_accuracy']:.4f}")
    print(f"Total de Amostras: {results_elec['global_metrics']['total_samples']}")
    print(f"Chunks Processados: {results_elec['global_metrics']['total_chunks_processed']}")
    
    # Plot
    plt.figure(figsize=(14, 5))
    plt.plot(range(1, len(results_elec['accuracies'])+1), results_elec['accuracies'],
             marker='o', linewidth=2, markersize=6, label='MCMO Adapter')
    plt.axhline(y=results_elec['mean_accuracy'], color='r', linestyle='--',
                label=f'Mean: {results_elec["mean_accuracy"]:.4f}')
    plt.xlabel('Chunk', fontsize=12)
    plt.ylabel('Accuracy', fontsize=12)
    plt.title('MCMO Adapter - Accuracy per Chunk (Electricity)', fontsize=14)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("\nElectricity dataset não disponível. Pulando teste.")

## 6. Comparação com Baseline Simples

In [ ]:
from skmultiflow.trees import HoeffdingTreeClassifier

print("=" * 70)
print("Teste 3: Comparação com Baseline (HoeffdingTree)")
print("=" * 70)

# Testar baseline simples (HoeffdingTree)
baseline = HoeffdingTreeClassifier()

print("\nTestando baseline em dados sintéticos...")
baseline_accuracies = []

for i, (X_chunk, y_chunk) in enumerate(zip(X_chunks, y_chunks)):
    predictions = []
    
    for j in range(len(X_chunk)):
        X_sample = X_chunk[j:j+1]
        y_sample = y_chunk[j:j+1]
        
        # Predict
        if i == 0 and j == 0:
            # Primeira amostra, predizer classe 0
            pred = 0
        else:
            pred = baseline.predict(X_sample)[0]
        
        predictions.append(pred)
        
        # Update
        baseline.partial_fit(X_sample, y_sample)
    
    accuracy = np.mean(np.array(predictions) == y_chunk)
    baseline_accuracies.append(accuracy)
    print(f"  Chunk {i+1}: Accuracy = {accuracy:.4f}")

baseline_mean = np.mean(baseline_accuracies)
print(f"\nBaseline Mean Accuracy: {baseline_mean:.4f}")
print(f"MCMO Mean Accuracy: {results['mean_accuracy']:.4f}")
print(f"\nDiferença: {(results['mean_accuracy'] - baseline_mean)*100:.2f} pontos percentuais")

if results['mean_accuracy'] > baseline_mean:
    print("✓ MCMO supera o baseline!")
else:
    print("⚠ MCMO não superou baseline neste teste")

In [ ]:
# Plot comparação
plt.figure(figsize=(14, 6))

plt.plot(range(1, len(results['accuracies'])+1), results['accuracies'],
         marker='o', linewidth=2, markersize=8, label='MCMO Adapter', color='blue')
plt.plot(range(1, len(baseline_accuracies)+1), baseline_accuracies,
         marker='s', linewidth=2, markersize=8, label='Baseline (HT)', color='orange')

plt.axhline(y=results['mean_accuracy'], color='blue', linestyle='--', alpha=0.5,
            label=f'MCMO Mean: {results["mean_accuracy"]:.4f}')
plt.axhline(y=baseline_mean, color='orange', linestyle='--', alpha=0.5,
            label=f'Baseline Mean: {baseline_mean:.4f}')

plt.xlabel('Chunk', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('MCMO Adapter vs Baseline - Accuracy per Chunk', fontsize=14)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Análise de Componentes MCMO

In [ ]:
print("=" * 70)
print("Teste 4: Análise de Componentes MCMO")
print("=" * 70)

# Testar diferentes valores de n_sources
print("\nTestando diferentes valores de n_sources...")

n_sources_values = [2, 3, 4, 5]
results_by_n_sources = {}

for n_sources in n_sources_values:
    print(f"\n--- Testing n_sources={n_sources} ---")
    
    res = test_mcmo_adapter(
        X_chunks=X_chunks[:8],  # Usar apenas 8 chunks para ser rápido
        y_chunks=y_chunks[:8],
        n_sources=n_sources,
        verbose=False
    )
    
    results_by_n_sources[n_sources] = res['mean_accuracy']
    print(f"Mean Accuracy: {res['mean_accuracy']:.4f}")

# Plot
plt.figure(figsize=(10, 6))
plt.bar(n_sources_values, 
        [results_by_n_sources[n] for n in n_sources_values],
        color='steelblue', alpha=0.7, edgecolor='black')
plt.xlabel('n_sources', fontsize=12)
plt.ylabel('Mean Accuracy', fontsize=12)
plt.title('MCMO Adapter - Impact of n_sources Parameter', fontsize=14)
plt.xticks(n_sources_values)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

best_n_sources = max(results_by_n_sources, key=results_by_n_sources.get)
print(f"\n✓ Melhor n_sources: {best_n_sources} (accuracy: {results_by_n_sources[best_n_sources]:.4f})")

## 8. Resumo dos Resultados

In [ ]:
print("="*70)
print("RESUMO DOS TESTES - MCMO ADAPTER")
print("="*70)

print("\n1. DADOS SINTÉTICOS")
print("-" * 70)
print(f"   MCMO Adapter:         {results['mean_accuracy']:.4f}")
print(f"   Baseline (HT):        {baseline_mean:.4f}")
print(f"   Diferença:            {(results['mean_accuracy'] - baseline_mean)*100:+.2f} p.p.")

if electricity_data is not None:
    print("\n2. ELECTRICITY DATASET")
    print("-" * 70)
    print(f"   MCMO Adapter:         {results_elec['mean_accuracy']:.4f}")
    print(f"   Total de samples:     {results_elec['global_metrics']['total_samples']}")

print("\n3. ANÁLISE DE PARÂMETROS")
print("-" * 70)
print(f"   Melhor n_sources:     {best_n_sources}")
print(f"   Accuracy com best:    {results_by_n_sources[best_n_sources]:.4f}")

print("\n4. CONCLUSÕES")
print("-" * 70)
if results['mean_accuracy'] > baseline_mean:
    print("   ✓ MCMO Adapter demonstrou superioridade sobre baseline simples")
else:
    print("   ⚠ MCMO Adapter não superou baseline (pode precisar de tuning)")

print("\n   Temporal splitting funcionou corretamente")
print("   Adapter está pronto para integração no pipeline principal")

print("\n5. PRÓXIMOS PASSOS")
print("-" * 70)
print("   1. Integrar MCMO em main.py (baseline_models)")
print("   2. Executar Phase 3 experiments (5 datasets)")
print("   3. Calcular rankings (Friedman + Wilcoxon)")
print("   4. Atualizar paper com resultados")

print("\n" + "="*70)

## 9. Exportar Resultados

In [ ]:
# Salvar resultados em CSV
results_df = pd.DataFrame({
    'chunk': range(1, len(results['accuracies'])+1),
    'mcmo_accuracy': results['accuracies'],
    'baseline_accuracy': baseline_accuracies
})

output_path = os.path.join(DSL_PATH, 'mcmo_test_results.csv')
results_df.to_csv(output_path, index=False)
print(f"✓ Resultados salvos em: {output_path}")

# Salvar também no Colab local
results_df.to_csv('mcmo_test_results.csv', index=False)
print("✓ Cópia salva em: /content/mcmo_test_results.csv")

print("\nPara baixar o arquivo:")
print("  - No Colab: Files → mcmo_test_results.csv → Download")
print("  - No Drive: Veja em DSL-AG-hybrid/mcmo_test_results.csv")

## 10. Informações Adicionais

In [ ]:
# Info sobre o adapter
print("Informações do MCMO Adapter")
print("="*70)

adapter = MCMOAdapter(n_sources=3, verbose=False)
info = adapter.get_info()

print(f"n_sources:           {info['n_sources']}")
print(f"initial_beach:       {info['initial_beach']}")
print(f"buffer_size:         {info['buffer_size']}")
print(f"using_mcmo:          {info['using_mcmo']}")

print("\nEstrutura do módulo mcmo:")
print("-"*70)
for f in os.listdir(MCMO_PATH):
    filepath = os.path.join(MCMO_PATH, f)
    if os.path.isfile(filepath):
        size = os.path.getsize(filepath)
        print(f"  {f:<25} {size:>10} bytes")

print("\n" + "="*70)
print("Teste completo! ✓")
print("="*70)